# UMUD Challenge — Submission 7: Thin-Line Aware Dual U-Net

## Why Previous Submissions Failed (What the Logs Told Us)
From Sub 6 training logs:
- **Apo model val_dice = 0.83** — Great, aponeurosis segmentation works well
- **Fascicle model val_dice = 0.29** — TERRIBLE. This is the root cause of all poor scores.

Fascicle lines are literally 1-2 pixels wide in the original images. When resized to 384px,
standard BCE+Dice loss treats them as almost-empty masks and the model learns to predict
nothing (all zeros = low loss). PA is derived from fascicle masks, so bad fascicle = bad PA = bad FL.

## Sub 7 Fixes (Each Targeted at the Specific Root Cause)
1. **Dilated GT masks for fascicle training** — We dilate the ground truth fascicle mask by 5px
   before feeding to the model. This makes thin lines learnable. At inference, Hough transform
   handles the now-thicker predicted lines correctly.
2. **Heavily weighted positive loss** — Fascicle pixels are extremely rare (~1-3% of image).
   We weight them 15x higher than background pixels in the loss function.
3. **Hough Line Transform for PA** — Instead of PCA on broken connected components,
   HoughLinesP is specifically designed to detect line angles in sparse/fragmented images.
   Much more robust to incomplete mask predictions.
4. **ResNet34 encoder (back to our best)** — Sub 3 (0.86) used ResNet34. Sub 6 tried VGG16
   and got 1.16. We go back to what actually works.

**Instructions:** GPU ON, Internet ON, Run All.


In [ ]:
%%capture
!pip install segmentation-models-pytorch albumentations


In [ ]:
import os, sys, math, glob, warnings, logging
os.environ["OPENCV_LOG_LEVEL"] = "SILENT"
os.environ["PYTHONWARNINGS"] = "ignore"
warnings.filterwarnings("ignore")
logging.disable(logging.CRITICAL)

import cv2
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from pathlib import Path
from tqdm.auto import tqdm

import segmentation_models_pytorch as smp
import albumentations as A
from albumentations.pytorch import ToTensorV2

SEED = 42
torch.manual_seed(SEED); np.random.seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device : {DEVICE}")
if torch.cuda.is_available():
    print(f"GPU    : {torch.cuda.get_device_name(0)}")

# --- Path Discovery ---
INPUT = Path("/kaggle/input")

def _find_dir(root, name):
    for p in root.rglob("*"):
        if p.is_dir() and p.name == name:
            return p
    return None

COMP = None
for candidate in INPUT.iterdir():
    if (candidate / "apo_imgs_v1").exists():
        COMP = candidate
        break
    if _find_dir(candidate, "apo_imgs_v1"):
        COMP = _find_dir(candidate, "apo_imgs_v1").parent
        break

if COMP is None:
    COMP = INPUT

APO_IMG_DIR   = COMP / "apo_imgs_v1"  / "apo_images_new_model_v1"
APO_MASK_DIR  = COMP / "apo_masks_v1" / "apo_masks_new_model_v1"
FASC_IMG_DIR  = COMP / "fasc_imgs_v1" / "fasc_images_new_model_v1"
FASC_MASK_DIR = COMP / "fasc_masks_v1"/ "fasc_masks_new_model_v1"
TEST_DIR      = COMP / "test_images_v2" / "test_set_v2"
OUT_DIR       = Path("/kaggle/working")

print("\n--- Path Check ---")
for name, p in [("APO imgs", APO_IMG_DIR), ("APO masks", APO_MASK_DIR),
                ("FASC imgs", FASC_IMG_DIR), ("FASC masks", FASC_MASK_DIR),
                ("TEST", TEST_DIR)]:
    ok = "OK" if p.exists() else "MISSING!"
    n  = len(list(p.rglob("*.*"))) if p.exists() else 0
    print(f"  [{ok}] {name}: {p} ({n} files)")

# --- Hyperparameters ---
IMG_SIZE           = 384
BATCH_SIZE         = 8
APO_EPOCHS         = 25
FASC_EPOCHS        = 30   # Fascicle needs more time
PATIENCE           = 7
LR                 = 5e-4
FASC_POS_WEIGHT    = 15.0  # KEY: weight positive (fascicle) pixels 15x more in loss
FASC_DILATION_PX   = 5     # KEY: dilate fascicle GT masks by 5px for learnability
SCALE_MM_PER_PIXEL = 0.06979


In [ ]:
_EXTS = {".tif", ".tiff", ".png", ".jpg", ".jpeg", ".bmp"}

TRAIN_AUG = A.Compose([
    A.Resize(IMG_SIZE, IMG_SIZE),
    A.HorizontalFlip(p=0.5),
    A.ShiftScaleRotate(shift_limit=0.04, scale_limit=0.06, rotate_limit=8,
                       border_mode=cv2.BORDER_CONSTANT, value=0, p=0.5),
    A.RandomBrightnessContrast(brightness_limit=0.12, contrast_limit=0.12, p=0.4),
    A.GaussNoise(p=0.3),
    A.Normalize(mean=[0.5], std=[0.5]),
    ToTensorV2(),
])

VAL_AUG = A.Compose([
    A.Resize(IMG_SIZE, IMG_SIZE),
    A.Normalize(mean=[0.5], std=[0.5]),
    ToTensorV2(),
])

class SegDataset(Dataset):
    def __init__(self, img_dir: Path, mask_dir: Path, aug, dilate_px=0):
        self.aug = aug
        self.dilate_px = dilate_px  # 0 for apo, FASC_DILATION_PX for fascicle
        # Build stem -> mask path lookup (handles _apo, _fasc, _mask suffixes)
        self._mask_lut = {}
        for p in mask_dir.rglob("*"):
            if p.is_file() and p.suffix.lower() in _EXTS:
                clean = p.stem.replace("_apo","").replace("_fasc","").replace("_mask","")
                self._mask_lut[clean] = p
        self.imgs = sorted([p for p in img_dir.rglob("*")
                            if p.is_file() and p.suffix.lower() in _EXTS])
        print(f"  Dataset: {len(self.imgs)} images, {len(self._mask_lut)} masks")

    def __len__(self): return len(self.imgs)

    def __getitem__(self, idx):
        ip = self.imgs[idx]
        clean = ip.stem.replace("_apo","").replace("_fasc","").replace("_mask","")
        mp = self._mask_lut.get(clean, None)

        img = cv2.imread(str(ip), cv2.IMREAD_GRAYSCALE)
        if img is None: img = np.zeros((800,1200), dtype=np.uint8)

        if mp is not None:
            msk = cv2.imread(str(mp), cv2.IMREAD_GRAYSCALE)
            if msk is None: msk = np.zeros(img.shape[:2], dtype=np.uint8)
            if msk.shape[:2] != img.shape[:2]:
                msk = cv2.resize(msk, (img.shape[1], img.shape[0]),
                                 interpolation=cv2.INTER_NEAREST)
        else:
            msk = np.zeros(img.shape[:2], dtype=np.uint8)

        msk = (msk > 127).astype(np.uint8)

        # KEY FIX: Dilate thin fascicle lines so the model can learn them!
        if self.dilate_px > 0:
            kernel = np.ones((self.dilate_px * 2 + 1, self.dilate_px * 2 + 1), np.uint8)
            msk = cv2.dilate(msk, kernel)

        out = self.aug(image=img, mask=msk)
        return out["image"], out["mask"].float().unsqueeze(0)  # (1,H,W), (1,H,W)


def make_loaders(img_dir, mask_dir, dilate_px=0):
    # Build index once to count
    probe = SegDataset(img_dir, mask_dir, aug=None, dilate_px=0)
    n = len(probe)
    val_n = max(1, int(n * 0.15))
    idx = np.random.RandomState(SEED).permutation(n).tolist()
    train_idx, val_idx = idx[val_n:], idx[:val_n]

    train_ds = SegDataset(img_dir, mask_dir, aug=TRAIN_AUG, dilate_px=dilate_px)
    val_ds   = SegDataset(img_dir, mask_dir, aug=VAL_AUG,   dilate_px=dilate_px)
    # Subset using same split
    train_ds.imgs = [probe.imgs[i] for i in train_idx]
    val_ds.imgs   = [probe.imgs[i] for i in val_idx]
    train_ds._mask_lut = val_ds._mask_lut = probe._mask_lut

    tl = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                    num_workers=2, pin_memory=True, drop_last=True)
    vl = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False,
                    num_workers=2, pin_memory=True)
    return tl, vl

print("Building Apo loaders (no dilation)...")
apo_tl, apo_vl = make_loaders(APO_IMG_DIR, APO_MASK_DIR, dilate_px=0)

print("Building Fascicle loaders (5px dilation - makes thin lines learnable)...")
fsc_tl, fsc_vl = make_loaders(FASC_IMG_DIR, FASC_MASK_DIR, dilate_px=FASC_DILATION_PX)

print(f"\nApo  -> train={len(apo_tl.dataset)} val={len(apo_vl.dataset)}")
print(f"Fasc -> train={len(fsc_tl.dataset)} val={len(fsc_vl.dataset)}")


In [ ]:
# ResNet34 back — proven to work (Sub 3 got 0.86, best score ever)
def build_resnet34_unet():
    return smp.Unet(
        encoder_name    = "resnet34",
        encoder_weights = "imagenet",
        in_channels     = 1,   # Grayscale directly — no need to convert to RGB
        classes         = 1,
    ).to(DEVICE)


class WeightedBceDiceLoss(nn.Module):
    # BCE+Dice with configurable positive weight for extreme class imbalance
    # Fascicle lines are ~1-3% of pixels, so we weight them 15x higher
    def __init__(self, pos_weight=1.0, dice_w=0.5, smooth=1.0):
        super().__init__()
        self.pos_weight = torch.tensor([pos_weight])
        self.dice_w = dice_w
        self.smooth = smooth

    def forward(self, logits, targets):
        pw = self.pos_weight.to(logits.device)
        bce = nn.functional.binary_cross_entropy_with_logits(
            logits, targets, pos_weight=pw)
        p = torch.sigmoid(logits)
        inter = (p * targets).sum(dim=(1,2,3))
        union = p.sum(dim=(1,2,3)) + targets.sum(dim=(1,2,3))
        dice  = 1 - (2*inter + self.smooth) / (union + self.smooth)
        return (1 - self.dice_w) * bce + self.dice_w * dice.mean()

print("ResNet34-UNet builder and WeightedBceDiceLoss defined.")
print(f"  Fascicle positive pixel weight = {FASC_POS_WEIGHT}x")


In [ ]:
def train_one(tag, train_dl, val_dl, max_epochs, pos_weight=1.0):
    model = build_resnet34_unet()
    opt   = optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
    sched = optim.lr_scheduler.CosineAnnealingLR(opt, T_max=max_epochs, eta_min=1e-6)
    crit  = WeightedBceDiceLoss(pos_weight=pos_weight, dice_w=0.5)
    scaler = torch.amp.GradScaler("cuda")
    best_dice, patience_cnt = -1.0, 0
    save_path = OUT_DIR / f"{tag}_best.pth"

    print(f"\n{'='*55}")
    print(f"  Training  {tag}  [{max_epochs} epochs, patience={PATIENCE}]")
    print(f"  pos_weight={pos_weight:.1f}x")
    print(f"{'='*55}")

    for epoch in range(1, max_epochs + 1):
        # Train
        model.train()
        t_loss = 0.0
        for imgs, masks in train_dl:
            imgs, masks = imgs.to(DEVICE), masks.to(DEVICE)
            opt.zero_grad()
            with torch.amp.autocast("cuda"):
                loss = crit(model(imgs), masks)
            scaler.scale(loss).backward()
            scaler.step(opt)
            scaler.update()
            t_loss += loss.item() * imgs.size(0)
        t_loss /= len(train_dl.dataset)
        sched.step()

        # Validate
        model.eval()
        dice_sum = 0.0
        with torch.no_grad():
            for imgs, masks in val_dl:
                imgs, masks = imgs.to(DEVICE), masks.to(DEVICE)
                with torch.amp.autocast("cuda"):
                    p = torch.sigmoid(model(imgs))
                pred  = (p > 0.5).float()
                inter = (pred * masks).sum(dim=(1,2,3))
                union = pred.sum(dim=(1,2,3)) + masks.sum(dim=(1,2,3))
                dice_sum += ((2*inter+1)/(union+1)).mean().item() * imgs.size(0)
        val_dice = dice_sum / len(val_dl.dataset)
        lr_now   = opt.param_groups[0]["lr"]

        print(f"  Ep {epoch:02d}  loss={t_loss:.4f}  val_dice={val_dice:.4f}  lr={lr_now:.2e}")

        if val_dice > best_dice:
            best_dice = val_dice
            patience_cnt = 0
            torch.save(model.state_dict(), save_path)
        else:
            patience_cnt += 1
            if patience_cnt >= PATIENCE:
                print(f"  Early stop at epoch {epoch}  (best_dice={best_dice:.4f})")
                break

    model.load_state_dict(torch.load(save_path, weights_only=True))
    model.eval()
    print(f"  {tag} done — best val_dice={best_dice:.4f}  saved->{save_path}")
    return model

# Apo model: standard loss (apo masks are thick/clear — no special weighting needed)
apo_model  = train_one("apo_r34",  apo_tl, apo_vl, max_epochs=APO_EPOCHS, pos_weight=1.0)

# Fascicle model: heavy positive weighting + more epochs
fasc_model = train_one("fasc_r34", fsc_tl, fsc_vl, max_epochs=FASC_EPOCHS, pos_weight=FASC_POS_WEIGHT)


In [ ]:
def measure_geometry_hough(apo_mask, fasc_mask, scale=SCALE_MM_PER_PIXEL):
    # PA: Hough Line Transform (designed for sparse/broken lines) - more robust than PCA
    # MT: Proven aponeurosis gap approach from Sub 3
    # FL: Trigonometry

    # === 1. PENNATION ANGLE via Hough Line Transform ===
    pa_deg = 15.0  # fallback
    if fasc_mask.max() > 0:
        # HoughLinesP finds line segments in sparse binary images — perfect for fascicles
        lines = cv2.HoughLinesP(
            fasc_mask,
            rho=1,
            theta=np.pi/180,
            threshold=30,        # min votes to be considered a line
            minLineLength=30,    # min segment length in pixels
            maxLineGap=20        # max gap to connect broken segments
        )
        if lines is not None and len(lines) > 0:
            angles, lengths = [], []
            for line in lines:
                x1, y1, x2, y2 = line[0]
                length = math.sqrt((x2-x1)**2 + (y2-y1)**2)
                if length < 10: continue
                # Angle from horizontal (fascicle inclination)
                angle = abs(math.degrees(math.atan2(abs(y2-y1), abs(x2-x1))))
                angles.append(angle)
                lengths.append(length)
            if angles:
                # Weighted average by segment length
                pa_deg = float(np.average(angles, weights=lengths))
        else:
            # Fallback: PCA on fascicle pixels if Hough finds nothing
            pts = np.column_stack(np.where(fasc_mask > 0)).astype(np.float64)
            if len(pts) >= 10:
                cov = np.cov(pts.T)
                if cov.shape == (2,2):
                    evals, evecs = np.linalg.eigh(cov)
                    v = evecs[:, np.argmax(evals)]
                    pa_deg = float(np.degrees(np.arctan2(abs(v[0]), abs(v[1]))))

    # === 2. MUSCLE THICKNESS from aponeurosis gap ===
    mt_mm = 20.0  # fallback
    n_apo, alabs, astats, _ = cv2.connectedComponentsWithStats(apo_mask, 8)
    if n_apo >= 3:
        top2 = sorted(range(1, n_apo),
                      key=lambda i: astats[i, cv2.CC_STAT_AREA], reverse=True)[:2]
        c1, c2 = top2[0], top2[1]
        y1 = np.mean(np.where(alabs == c1)[0])
        y2 = np.mean(np.where(alabs == c2)[0])
        sup_c, deep_c = (c1,c2) if y1 < y2 else (c2,c1)

        sm = (alabs == sup_c).astype(np.uint8)
        dm = (alabs == deep_c).astype(np.uint8)
        sup_bot  = np.where(sm.any(axis=0), sm.shape[0]-1-np.argmax(sm[::-1],axis=0), np.nan)
        deep_top = np.where(dm.any(axis=0), np.argmax(dm, axis=0), np.nan)
        thick = deep_top - sup_bot
        valid = ~np.isnan(thick) & (thick > 0)
        if valid.sum() > 50:
            mt_mm = float(np.nanmedian(thick[valid]) * scale)

    # === 3. FASCICLE LENGTH ===
    fl_mm = mt_mm / math.sin(math.radians(max(pa_deg, 3.0)))

    return (float(np.clip(pa_deg, 3.0, 45.0)),
            float(np.clip(fl_mm,  20.0, 200.0)),
            float(np.clip(mt_mm,  5.0,  40.0)))

print("measure_geometry_hough() defined.")


In [ ]:
_INF_AUG = A.Compose([
    A.Resize(IMG_SIZE, IMG_SIZE),
    A.Normalize(mean=[0.5], std=[0.5]),
    ToTensorV2(),
])

def predict_mask(model, img_gray, threshold=0.5, tta=True):
    # Predict binary mask with optional horizontal-flip TTA
    h0, w0 = img_gray.shape[:2]
    x = _INF_AUG(image=img_gray)["image"].unsqueeze(0).to(DEVICE)

    with torch.no_grad(), torch.amp.autocast("cuda"):
        p_norm = torch.sigmoid(model(x))
        if tta:
            p_flip = torch.sigmoid(model(torch.flip(x, dims=[3])))
            p_flip = torch.flip(p_flip, dims=[3])
            p_avg  = (p_norm + p_flip) / 2.0
        else:
            p_avg  = p_norm

    prob = p_avg.squeeze().cpu().float().numpy()
    prob_full = cv2.resize(prob, (w0, h0), interpolation=cv2.INTER_LINEAR)
    mask = (prob_full > threshold).astype(np.uint8) * 255

    # Light morphological clean-up to remove isolated speckle
    mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, np.ones((3,3), np.uint8))
    return mask


# Scan TEST_DIR directly — never rely on sample_submission row count!
test_files = sorted([p for p in TEST_DIR.rglob("*")
                     if p.is_file() and p.suffix.lower() in _EXTS])
print(f"Found {len(test_files)} test images.")

apo_model.eval(); fasc_model.eval()
results = []

for tp in tqdm(test_files, desc="Inference"):
    img = cv2.imread(str(tp), cv2.IMREAD_GRAYSCALE)
    if img is None:
        print(f"  WARN: cannot read {tp.name}")
        continue

    apo_mask  = predict_mask(apo_model,  img, tta=True)
    fasc_mask = predict_mask(fasc_model, img, tta=True)

    pa, fl, mt = measure_geometry_hough(apo_mask, fasc_mask)

    results.append({
        "image_id": tp.name,   # filename WITH extension e.g. IMG_00001.tif
        "pa_deg":   round(pa, 3),
        "fl_mm":    round(fl, 3),
        "mt_mm":    round(mt, 3),
    })

sub = pd.DataFrame(results)
out_path = OUT_DIR / "submission.csv"
sub.to_csv(out_path, sep=",", index=False)   # COMMA separator!

print(f"\nSubmission saved: {out_path}  ({len(sub)} rows)")
print(f"  PA  range: {sub.pa_deg.min():.1f} - {sub.pa_deg.max():.1f} deg")
print(f"  FL  range: {sub.fl_mm.min():.1f} - {sub.fl_mm.max():.1f} mm")
print(f"  MT  range: {sub.mt_mm.min():.1f} - {sub.mt_mm.max():.1f} mm")
print("\nFirst 5 rows:")
print(sub.head())
